In [ ]:
!pip install ultralytics supervision opencv-python-headless -q

In [ ]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display
import os

In [ ]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

In [ ]:
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

# Draw frame dimensions so you know coordinate range
h, w = frame.shape[:2]
print(f"Frame size: {w}x{h}")

# Save first frame for inspection
cv2.imwrite("first_frame.jpg", frame)
files.download("first_frame.jpg")

In [5]:
# Traffic light ROIs (small boxes around each light, 40x40px)
TRAFFIC_LIGHTS = [
    {"id": 1, "x": 266, "y": 464, "w": 40, "h": 40},
    {"id": 2, "x": 1002, "y": 364, "w": 40, "h": 40},
    {"id": 3, "x": 1167, "y": 506, "w": 40, "h": 40},
]

# Stop lines as (x1,y1,x2,y2)
STOP_LINES = [
    {"id": 1, "line": (304, 1042, 1893, 831)},   # our side
    {"id": 2, "line": (328, 669, 1168, 660)},     # other side
]

VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

In [6]:
def get_light_state(frame, light_roi):
    x, y, w, h = light_roi["x"], light_roi["y"], light_roi["w"], light_roi["h"]
    roi = frame[y:y+h, x:x+w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # Red mask (two ranges because red wraps in HSV)
    red1 = cv2.inRange(hsv, np.array([0,100,100]), np.array([10,255,255]))
    red2 = cv2.inRange(hsv, np.array([160,100,100]), np.array([180,255,255]))
    red_mask = red1 | red2

    # Green mask
    green_mask = cv2.inRange(hsv, np.array([40,100,100]), np.array([90,255,255]))

    red_pixels = cv2.countNonZero(red_mask)
    green_pixels = cv2.countNonZero(green_mask)

    if red_pixels > green_pixels and red_pixels > 10:
        return "red"
    elif green_pixels > red_pixels and green_pixels > 10:
        return "green"
    else:
        return "unknown"

In [7]:
def is_crossing_line(bbox, line):
    x1, y1, x2, y2 = line
    bx1, by1, bx2, by2 = bbox
    vehicle_bottom_y = by2
    vehicle_center_x = (bx1 + bx2) / 2

    # Line equation: y = y1 + (y2-y1)/(x2-x1) * (x - x1)
    if x2 == x1:
        return False
    line_y_at_vehicle_x = y1 + (y2 - y1) / (x2 - x1) * (vehicle_center_x - x1)

    return vehicle_bottom_y >= line_y_at_vehicle_x

In [ ]:
model = YOLO("yolov8m.pt")
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

output_path = "output_redlight.mp4"
violations_dir = "violations"
os.makedirs(violations_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

logged_violations = set()  # track (vehicle_id, line_id) to avoid duplicate logs
violation_log = []
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect traffic light states
    light_states = [get_light_state(frame, l) for l in TRAFFIC_LIGHTS]
    any_red = "red" in light_states

    # Detect and track vehicles
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    mask = [int(cls) in VEHICLE_CLASSES for cls in detections.class_id]
    detections = detections[mask]
    detections = tracker.update_with_detections(detections)

    # Check violations
    if any_red and detections.tracker_id is not None:
        for i, tracker_id in enumerate(detections.tracker_id):
            bbox = detections.xyxy[i]
            for stop_line in STOP_LINES:
                key = (tracker_id, stop_line["id"])
                if key not in logged_violations and is_crossing_line(bbox, stop_line["line"]):
                    logged_violations.add(key)
                    snapshot_path = f"{violations_dir}/violation_{tracker_id}_{stop_line['id']}_{frame_count}.jpg"
                    cv2.imwrite(snapshot_path, frame)
                    violation_log.append({
                        "frame": frame_count,
                        "tracker_id": int(tracker_id),
                        "stop_line_id": stop_line["id"],
                        "snapshot": snapshot_path
                    })
                    print(f"VIOLATION: Vehicle {tracker_id} crossed line {stop_line['id']} at frame {frame_count}")

    # Draw stop lines
    for sl in STOP_LINES:
        x1,y1,x2,y2 = sl["line"]
        cv2.line(frame, (x1,y1), (x2,y2), (0,0,255), 2)

    # Draw traffic light ROIs with state
    for i, light in enumerate(TRAFFIC_LIGHTS):
        state = light_states[i]
        color = (0,0,255) if state=="red" else (0,255,0) if state=="green" else (128,128,128)
        cv2.rectangle(frame, (light["x"], light["y"]),
                      (light["x"]+light["w"], light["y"]+light["h"]), color, 2)
        cv2.putText(frame, state, (light["x"], light["y"]-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Annotate vehicles
    if detections.tracker_id is not None:
        labels = [
            f"ID{tid} {VEHICLE_CLASSES.get(int(cls),'v')} {conf:.2f}"
            for tid, cls, conf in zip(detections.tracker_id, detections.class_id, detections.confidence)
        ]
        frame = box_annotator.annotate(scene=frame, detections=detections)
        frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    out.write(frame)
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print(f"\nDone. Total violations: {len(violation_log)}")

In [ ]:
display(Video(output_path, embed=True, width=800))

In [ ]:
import zipfile

# Zip violation snapshots
with zipfile.ZipFile("violations.zip", "w") as zf:
    for f in os.listdir(violations_dir):
        zf.write(os.path.join(violations_dir, f), f)

files.download(output_path)
files.download("violations.zip")